In [ ]:
import numpy as np, pandas as pd, random
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score

SEED = 0
random.seed(SEED); np.random.seed(SEED)

x = np.array([0., 1., 2., 3., 4.])
y = np.array([10., 8., 7., 4., 1.])
N = len(x)

def sums(x, y):
    xb, yb = x.mean(), y.mean()
    return xb, yb, ((x - xb) ** 2).sum(), ((x - xb) * (y - yb)).sum()

xb, yb, Sxx, Sxy = sums(x, y)
print(f"x_bar={xb}, y_bar={yb}, Sxx={Sxx}, Sxy={Sxy}")

x_bar=2.0, y_bar=6.0, Sxx=10.0, Sxy=-22.0


In [ ]:
w1 = Sxy / Sxx
w0 = yb - w1 * xb
pred = w0 + w1 * x
sse = ((y - pred) ** 2).sum()
mse = sse / N
sst = ((y - yb) ** 2).sum()
r2 = 1 - sse / sst
print(f"OLS: w0={w0:.4f}, w1={w1:.4f}, SSE={sse:.4f}, MSE={mse:.4f}, SST={sst:.1f}, R2={r2:.4f}")

lr = LinearRegression().fit(x.reshape(-1, 1), y)
print("sklearn OLS:", lr.intercept_, lr.coef_[0],
      mean_squared_error(y, lr.predict(x.reshape(-1, 1))), r2_score(y, lr.predict(x.reshape(-1, 1))))

OLS: w0=10.4000, w1=-2.2000, SSE=1.6000, MSE=0.3200, SST=50.0, R2=0.9680
sklearn OLS: 10.399999999999999 -2.1999999999999997 0.3200000000000001 0.968


In [ ]:
rows = []
for lam in [0, 1, 10, 100, 1e4, 1e8]:
    w1r = Sxy / (Sxx + lam)
    w0r = yb - w1r * xb
    sk = Ridge(alpha=lam).fit(x.reshape(-1, 1), y) if lam > 0 else lr
    rows.append(dict(lam=lam, w1_hand=w1r, w0_hand=w0r, w1_sklearn=sk.coef_[0], w0_sklearn=sk.intercept_))
pd.DataFrame(rows)

,lam,w1_hand,w0_hand,w1_sklearn,w0_sklearn
0,0.0,-2.200000e+00,10.400000,-2.200000e+00,10.400000
1,1.0,-2.000000e+00,10.000000,-2.000000e+00,10.000000
2,10.0,-1.100000e+00,8.200000,-1.100000e+00,8.200000
3,100.0,-2.000000e-01,6.400000,-2.000000e-01,6.400000
4,10000.0,-2.197802e-03,6.004396,-2.197802e-03,6.004396
5,100000000.0,-2.200000e-07,6.000000,-2.200000e-07,6.000000


In [ ]:
xc = x / 10.0
xb_c, yb_c, Sxx_c, Sxy_c = sums(xc, y)
print(f"cm: Sxx={Sxx_c}, Sxy={Sxy_c}")

w1_c = Sxy_c / Sxx_c
w0_c = yb_c - w1_c * xb_c
pred_c = w0_c + w1_c * xc
print("OLS slope per cm:", w1_c, "| per mm:", w1_c / 10)
print("OLS predictions identical:", np.allclose(pred, pred_c), pred_c)

lam = 1.0
w1_rc = Sxy_c / (Sxx_c + lam)
print("Ridge(lambda=1) slope per cm:", w1_rc, "-> per mm:", w1_rc / 10,
      "| mm-data slope was", Sxy / (Sxx + lam))
w0_rc = yb_c - w1_rc * xb_c
print("Ridge predictions differ:", np.round(w0_rc + w1_rc * xc, 3), "vs", np.round(yb - Sxy/(Sxx+lam)*xb + Sxy/(Sxx+lam)*x, 3))

cm: Sxx=0.1, Sxy=-2.2
OLS slope per cm: -22.0 | per mm: -2.2
OLS predictions identical: True [10.4  8.2  6.   3.8  1.6]
Ridge(lambda=1) slope per cm: -2.0 -> per mm: -0.2 | mm-data slope was -2.0
Ridge predictions differ: [6.4 6.2 6.  5.8 5.6] vs [10.  8.  6.  4.  2.]


In [ ]:
def gd(X, y, step, max_iter=100000, tol=1e-8):
    w = np.zeros(X.shape[1])
    for k in range(1, max_iter + 1):
        g = -X.T @ (y - X @ w)
        w_new = w - step * g
        if not np.all(np.isfinite(w_new)) or np.linalg.norm(w_new) > 1e12:
            return None, k, "diverged"
        if np.linalg.norm(w_new - w) < tol:
            return w_new, k, "converged"
        w = w_new
    return w, max_iter, "max_iter"

def design(v):
    return np.c_[np.ones_like(v), v]

res = []
for name, v in [("mm", x), ("cm", xc), ("standardised", (x - x.mean()) / x.std())]:
    X = design(v)
    ev = np.linalg.eigvalsh(X.T @ X)
    L, mu = ev.max(), ev.min()
    _, it, st = gd(X, y, 1 / L)
    _, it_big, st_big = gd(X, y, 2.5 / L)
    res.append(dict(unit=name, L=L, mu=mu, kappa=L / mu, iters_step_1_over_L=it, status=st, status_step_2p5_over_L=st_big))
pd.DataFrame(res)

,unit,L,mu,kappa,iters_step_1_over_L,status,status_step_2p5_over_L
0,mm,33.507811,1.492189,22.455467,389,converged,diverged
1,cm,5.203919,0.096081,54.161537,945,converged,diverged
2,standardised,5.000000,5.000000,1.000000,2,converged,diverged
